# Entrenamiento y análisis del modelo LSTM

Notebook independiente para entrenar la LSTM bidireccional con los landmarks existentes, guardar su checkpoint en Google Drive y revisar curvas y resultados de prueba. El entrenamiento reutiliza el flujo común: separación por participante, normalización, aumentación solo en train, parada temprana y selección por pérdida de validación.

## Google Colab y Drive

Copia la carpeta `landmark_recognition` actualizada a `Mi unidad/FASE 2 IA/Fase 2/Codigo/landmark_recognition` o `Mi unidad/landmark_recognition`, abre este notebook en Colab y ejecuta las celdas en orden. Si todavía no existe `data/landmarks_dataset.npz`, coloca `LANDMARKS.zip` en una de las rutas detectadas por la primera celda. Este notebook no modifica ningún archivo de SPOTER.

In [1]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

from google.colab import drive
drive.mount('/content/drive')

MYDRIVE = Path('/content/drive/MyDrive')
PROJECT_CANDIDATES = [
    MYDRIVE / 'FASE 2 IA' / 'Fase 2' / 'Codigo' / 'landmark_recognition',
    MYDRIVE / 'landmark_recognition',
]
PROJECT_DIR = next((path for path in PROJECT_CANDIDATES if path.is_dir()), None)
if PROJECT_DIR is None:
    raise FileNotFoundError(
        'No se encontró landmark_recognition en Drive. Copia la carpeta completa y vuelve a ejecutar.'
    )

ARCHIVE_CANDIDATES = [
    MYDRIVE / 'LANDMARKS.zip',
    MYDRIVE / 'FASE 2 IA' / 'Fase 2' / 'LANDMARKS.zip',
    PROJECT_DIR / 'LANDMARKS.zip',
]
ARCHIVE_PATH = next((path for path in ARCHIVE_CANDIDATES if path.is_file()), None)
DATASET_PATH = PROJECT_DIR / 'data' / 'landmarks_dataset.npz'
CHECKPOINT_PATH = PROJECT_DIR / 'checkpoints' / 'lstm_subject.pth'
sys.path.insert(0, str(PROJECT_DIR.parent))

print(f'Proyecto: {PROJECT_DIR}')
print(f'Dataset preparado: {DATASET_PATH.is_file()}')
print(f'ZIP de landmarks encontrado: {ARCHIVE_PATH is not None}')

Mounted at /content/drive
Proyecto: /content/drive/MyDrive/FASE 2 IA/Fase 2/Codigo/landmark_recognition
Dataset preparado: True
ZIP de landmarks encontrado: True


## 1. Preparar y cargar los landmarks

Cada muestra debe tener forma `[64 frames, 75 puntos, 3 coordenadas]`; las etiquetas son códigos numéricos si no agregas un mapeo de clases.

In [2]:
from landmark_recognition.prepare_landmark_dataset import prepare_dataset

if not DATASET_PATH.is_file():
    if ARCHIVE_PATH is None:
        raise FileNotFoundError(
            f'Falta {DATASET_PATH} y no se encontró LANDMARKS.zip en las ubicaciones configuradas.'
        )
    prepare_dataset(ARCHIVE_PATH, DATASET_PATH, frame_count=64)

with np.load(DATASET_PATH, allow_pickle=False) as dataset:
    X = dataset['X']
    y = dataset['y'].astype(int)
    signers = dataset['signers'].astype(str)
    class_names = dataset['class_names'].astype(str)

print(f'X: {X.shape} [secuencias, frames, puntos, xyz]')
print(f'Clases: {len(class_names)}; participantes: {len(np.unique(signers))}')
class_counts = np.bincount(y, minlength=len(class_names))
print(f'Ejemplos por clase: mínimo={class_counts.min()}, máximo={class_counts.max()}')

X: (1000, 64, 75, 3) [secuencias, frames, puntos, xyz]
Clases: 50; participantes: 5
Ejemplos por clase: mínimo=20, máximo=20


## 2. Entrenar LSTM

Por defecto se reserva un participante distinto para validación y prueba. Esto mide generalización a una persona no vista y puede ser difícil con pocos participantes; no cambies a `random` para reportar generalización. El split aleatorio mezcla personas y sirve solo como diagnóstico. El checkpoint LSTM queda en Drive y no reemplaza el Transformer.

In [3]:
from types import SimpleNamespace
import importlib
import inspect
import landmark_recognition.data as landmark_data
import landmark_recognition.models as landmark_models
import landmark_recognition.lstm_recurrente.model as landmark_lstm_model
import landmark_recognition.train_model as landmark_train

# Recarga fuentes desde Drive para no entrenar con módulos antiguos en memoria.
importlib.reload(landmark_data)
importlib.reload(landmark_models)
importlib.reload(landmark_lstm_model)
importlib.reload(landmark_train)
print(f'Trainer cargado desde: {landmark_train.__file__}')
trainer_source = inspect.getsource(landmark_train.train)
for required in ('train_accuracy=', 'val_loss=', 'Early stopping at epoch'):
    if required not in trainer_source:
        raise RuntimeError(
            f'train_model.py parece desactualizado (falta {required!r}). '
            'Copia los archivos actuales desde Codigo a Drive y reinicia el runtime.'
        )

EPOCHS = 100
BATCH_SIZE = 16
LEARNING_RATE = 0.001
SPLIT_MODE = 'subject'  # Usa 'random' solo para diagnóstico de aprendizaje.
CHECKPOINT_PATH = PROJECT_DIR / 'checkpoints' / f'lstm_{SPLIT_MODE}.pth'

landmark_train.train(SimpleNamespace(
    dataset=DATASET_PATH,
    architecture='lstm',
    output=CHECKPOINT_PATH,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    validation_signer='',
    test_signer='',
    split_mode=SPLIT_MODE,
    no_augmentation=False,
    seed=42,
    early_stopping_patience=15,
    early_stopping_min_delta=0.01,
))

Trainer cargado desde: /content/drive/MyDrive/FASE 2 IA/Fase 2/Codigo/landmark_recognition/train_model.py
Split mode=subject; train=600, validation=200, test=200
Participants: train=['0000', '0001', '0002'], validation=['0003'], test=['0004']
train class counts: min=12, max=12, missing=[]
validation class counts: min=4, max=4, missing=[]
test class counts: min=4, max=4, missing=[]
Training augmentation: enabled (online; labels unchanged)
Epoch 001/100: loss=4.2300, train_accuracy=0.0350, val_loss=4.0881, val_accuracy=0.0250, lr=0.001
Epoch 002/100: loss=4.1055, train_accuracy=0.0433, val_loss=4.0714, val_accuracy=0.0200, lr=0.001
Epoch 003/100: loss=4.0450, train_accuracy=0.0317, val_loss=4.0427, val_accuracy=0.0250, lr=0.001
Epoch 004/100: loss=3.9976, train_accuracy=0.0483, val_loss=4.0471, val_accuracy=0.0150, lr=0.001
Epoch 005/100: loss=3.9912, train_accuracy=0.0333, val_loss=4.1181, val_accuracy=0.0150, lr=0.001
Epoch 006/100: loss=3.9832, train_accuracy=0.0350, val_loss=4.0321, 

## 3. Curvas de entrenamiento y validación

Una brecha creciente entre entrenamiento y validación indica sobreajuste. Compara la pérdida y exactitud de validación junto con los resultados del conjunto de prueba; la exactitud de prueba no se usa para seleccionar épocas.

In [ ]:
import torch

if not CHECKPOINT_PATH.is_file():
    raise FileNotFoundError(f'No se guardó el checkpoint esperado: {CHECKPOINT_PATH}')
checkpoint = torch.load(CHECKPOINT_PATH, map_location='cpu', weights_only=True)
if checkpoint['architecture'] != 'lstm':
    raise ValueError(f'El checkpoint no es LSTM: {checkpoint["architecture"]!r}')
history = checkpoint['history']
epochs = np.arange(1, len(history['train_loss']) + 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(epochs, history['train_loss'], label='train')
axes[0].plot(epochs, history['validation_loss'], label='validación')
axes[0].set(title='Pérdida', xlabel='Época', ylabel='Cross entropy')
axes[0].grid(True, alpha=0.3)
axes[0].legend()
axes[1].plot(epochs, history['train_accuracy'], label='train')
axes[1].plot(epochs, history['validation_accuracy'], label='validación')
axes[1].set(title='Exactitud', xlabel='Época', ylabel='Accuracy', ylim=(0, 1))
axes[1].grid(True, alpha=0.3)
axes[1].legend()
fig.tight_layout()
plt.show()

print(f'Mejor época: {checkpoint["best_epoch"]}')
print(f'Pérdida de validación seleccionada: {checkpoint["validation_loss"]:.4f}')
print(f'Accuracy de validación: {checkpoint["validation_accuracy"]:.1%}')
print(f'Accuracy de prueba: {checkpoint["test_accuracy"]:.1%}')

## 4. Matriz de confusión de prueba

Las filas son las etiquetas reales y las columnas las predicciones. Con 50 clases, una matriz completa es grande; se muestran sus dimensiones y las 10 clases con mayor exactitud.

In [ ]:
targets = np.asarray(checkpoint['test_targets'], dtype=int)
predictions = np.asarray(checkpoint['test_predictions'], dtype=int)
if not len(targets):
    raise ValueError('El checkpoint no contiene resultados de prueba; revisa que el split tenga test.')

confusion = np.zeros((len(class_names), len(class_names)), dtype=int)
np.add.at(confusion, (targets, predictions), 1)
fig, ax = plt.subplots(figsize=(10, 8))
image = ax.imshow(confusion, cmap='Blues', interpolation='nearest', aspect='auto')
ax.set(title='Matriz de confusión (prueba)', xlabel='Predicción', ylabel='Etiqueta real')
fig.colorbar(image, ax=ax, label='Secuencias')
fig.tight_layout()
plt.show()

support = confusion.sum(axis=1)
per_class_accuracy = np.divide(
    confusion.diagonal(), support,
    out=np.full(len(class_names), np.nan), where=support > 0,
)
available = np.flatnonzero(support > 0)
best = available[np.argsort(per_class_accuracy[available])[-10:][::-1]]
for index in best:
    print(f'{class_names[index]}: accuracy={per_class_accuracy[index]:.1%}, ejemplos={support[index]}')

## Usar el checkpoint

Para inferencia local en el equipo que tiene conectada la webcam, usa el script OpenCV/MediaPipe desde la carpeta `Codigo`:

```powershell
python -m landmark_recognition.predict_video `
  --checkpoint landmark_recognition\checkpoints\lstm_subject.pth `
  --source 0
```

En Colab la webcam del navegador requiere el helper `colab_webcam.py`; instala MediaPipe compatible y reinicia el runtime si ya fue importado. La LSTM predice una ventana de frames, no traduce automáticamente una conversación continua.